In [ ]:
"""
----
data_merge.ipynb로 만든 dataset에서
라벨링 오류로 판단되는 이미지 376건을 검출하고 제외 목록을 저장한다.

검증 항목 (총 376건)
--------------------
1) bbox 위치 겹침       : 같은 이미지 내 서로 다른 bbox 쌍의 IoU > 0.3   -> 128건
2) 라벨 개수 불일치     : 파일명(K-코드 개수)과 실제 라벨 줄 수가 다름   -> 226건
3) 종횡비 이상치        : bbox width/height 비율이 0.2~5.0 범위를 벗어남 -> 13건
4) 알약 간 거리 이상치  : 같은 이미지 내 bbox 중심점 간 최소거리 < 0.05  -> 9건

"""

In [ ]:
import os
import json

BASE_PATH = "./data/dataset_combined_final"
CLASS_MAP_PATH = "./data/class_map_118.json"
OUTPUT_JSON = "./data/excluded_problem_images_v4final.json"


In [ ]:
def extract_combo_code(fname: str) -> str:
    parts = fname.replace(".png", "").replace(".txt", "").split("_")
    for p in parts:
        if p.startswith("K-"):
            return p
    return fname


def compute_iou(box1, box2) -> float:
    x1 = max(box1[0] - box1[2] / 2, box2[0] - box2[2] / 2)
    y1 = max(box1[1] - box1[3] / 2, box2[1] - box2[3] / 2)
    x2 = min(box1[0] + box1[2] / 2, box2[0] + box2[2] / 2)
    y2 = min(box1[1] + box1[3] / 2, box2[1] + box2[3] / 2)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

In [ ]:
def load_all_labels():
    img_id_to_boxes = {}
    img_id_to_fname = {}
    for split in ("train", "val"):
        label_dir = f"{BASE_PATH}/labels/{split}"
        for fname in os.listdir(label_dir):
            img_id = fname.replace(".txt", "")
            boxes = []
            with open(f"{label_dir}/{fname}") as f:
                for line in f:
                    parts = line.split()
                    cls_id = int(parts[0])
                    xc, yc, w, h = map(float, parts[1:])
                    boxes.append((cls_id, xc, yc, w, h))
            img_id_to_boxes[img_id] = boxes
            img_id_to_fname[img_id] = f"{img_id}.png"
    return img_id_to_boxes, img_id_to_fname


def find_bbox_overlap(img_id_to_boxes):
    problem_ids = set()
    for img_id, boxes in img_id_to_boxes.items():
        for i in range(len(boxes)):
            for j in range(i + 1, len(boxes)):
                iou = compute_iou(boxes[i][1:], boxes[j][1:])
                if iou > 0.3:
                    problem_ids.add(img_id)
    print(f"[검증 1] bbox 위치 겹침(IoU>0.3): {len(problem_ids)}개 이미지")
    return problem_ids

In [ ]:
def find_count_mismatch(img_id_to_boxes):
    problem_ids = set()
    for img_id, boxes in img_id_to_boxes.items():
        combo = extract_combo_code(img_id)
        expected_count = len(combo.replace("K-", "").split("-"))
        actual_count = len(set(b[0] for b in boxes))
        if expected_count != actual_count:
            problem_ids.add(img_id)
    print(f"[검증 2] 라벨 개수 불일치: {len(problem_ids)}개 이미지")
    return problem_ids

In [ ]:
def find_ratio_outliers(img_id_to_boxes):
    problem_ids = set()
    for img_id, boxes in img_id_to_boxes.items():
        for (_, xc, yc, w, h) in boxes:
            ratio = w / h if h > 0 else 0
            if ratio < 0.2 or ratio > 5.0:
                problem_ids.add(img_id)
                break
    print(f"[검증 3] 종횡비 이상치(0.2~5.0 밖): {len(problem_ids)}개 이미지")
    return problem_ids

In [ ]:
def compute_min_distance(centers):
    if len(centers) < 2:
        return None
    min_dist = float("inf")
    for i in range(len(centers)):
        for j in range(i + 1, len(centers)):
            x1, y1 = centers[i]
            x2, y2 = centers[j]
            dist = ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5
            min_dist = min(min_dist, dist)
    return min_dist


def find_distance_outliers(img_id_to_boxes):
    problem_ids = set()
    for img_id, boxes in img_id_to_boxes.items():
        centers = [(b[1], b[2]) for b in boxes]
        d = compute_min_distance(centers)
        if d is not None and d < 0.05:
            problem_ids.add(img_id)
    print(f"[검증 4] 알약 간 거리 이상치(<0.05): {len(problem_ids)}개 이미지")
    return problem_ids

In [ ]:
def main():
    img_id_to_boxes, img_id_to_fname = load_all_labels()
    print(f"전체 이미지 수: {len(img_id_to_boxes)}")

    p1 = find_bbox_overlap(img_id_to_boxes)
    p2 = find_count_mismatch(img_id_to_boxes)
    p3 = find_ratio_outliers(img_id_to_boxes)
    p4 = find_distance_outliers(img_id_to_boxes)

    excluded = p1 | p2 | p3 | p4
    print(f"\n최종 제외 대상(중복 제거): {len(excluded)}개")
    print(f"정제 전: {len(img_id_to_boxes)} -> 정제 후: {len(img_id_to_boxes) - len(excluded)}")

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(sorted(excluded), f, ensure_ascii=False, indent=2)
    print(f"\n제외 목록 저장 완료: {OUTPUT_JSON}")


if __name__ == "__main__":
    main()